In [ ]:
%reload_ext autoreload
%autoreload 2


In [ ]:
import jax
import jax.numpy as jnp

# Set device to CPU
# jax.config.update("jax_platform_name", "cpu")
jax.config.update("jax_enable_x64", True)

import matplotlib.pyplot as plt


# Sequential Monte Carlo (SMC)

This notebook shows how to use the new `probjax.inference.smc_runner.SMC` interface with the SMC kernels.


In [ ]:
from probjax.inference.mcmc import hmc
from probjax.inference.smc import smc
from probjax.inference.smc.path import GeometricPath
from probjax.inference.smc_runner import SMC


## 1D toy model
We define a simple prior and likelihood, then run tempered SMC.


In [ ]:
key = jax.random.PRNGKey(0)
num_particles = 256

def logprior_fn(x):
    return -0.5 * jnp.sum(x**2)

def loglikelihood_fn(x):
    return -0.5 * jnp.sum((x - 1.0) ** 2)

particles = jax.random.normal(key, (num_particles, 1))


In [ ]:
kernel = smc(
    path=GeometricPath(),
    logprior_fn=logprior_fn,
    loglikelihood_fn=loglikelihood_fn,
    mcmc_kernel=hmc,
    num_mcmc_steps=3,
    num_integration_steps=5,
)

state = kernel.init(particles, path=GeometricPath())
params = kernel.init_params(
    particles,
    path=GeometricPath(),
    logprior_fn=logprior_fn,
    loglikelihood_fn=loglikelihood_fn,
    mcmc_kernel=hmc,
    num_integration_steps=5,
    step_size=0.2,
)


In [ ]:
runner = SMC(kernel, verbose=True)
tempering_params = jnp.linspace(0.0, 1.0, 20)
final_state, final_params = runner.run(key, state, tempering_params, params)


In [ ]:
plt.hist(final_state.particles.flatten(), bins=50, density=True)
plt.title("SMC final particles")


## Sampling with history and tuning
The runner can also return the full particle and weight history and optionally tune MCMC parameters.


In [ ]:
particles_hist, weights_hist, final_state, final_params = runner.sample(
    key,
    state,
    tempering_params,
    params,
    tune_params=True,
)


In [ ]:
plt.plot(weights_hist.mean(axis=1))
plt.title("Mean weight over tempering steps")
